# DermAssist AI: ResNet18 baseline on Colab (Phase 3)

Trains the end-to-end ResNet18 baseline on the fixed lesion-level split, selects the best epoch on the **validation** set, and evaluates on validation.

> Educational and research use only. Not a diagnostic tool.

**Before you start**
* Runtime > Change runtime type > **GPU**.
* Run `colab_quickstart.ipynb` first: it creates `splits.csv` on Drive, which this notebook reuses.
* The test set is not touched here. It is evaluated once, in Phase 5, after all choices are final.

## 1. Check the hardware

In [ ]:
import platform
print("Python", platform.python_version())
try:
    import torch
    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("torch is not installed (it is pre-installed on Colab)")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Get the code

Two ways, pick one:

* **GitHub (default):** the code is cloned from `REPO_URL`. Re-running this cell pulls the latest code. If the repository is **private**, Colab needs a GitHub token: see `docs/COLAB_SETUP.md`.
* **Zip on Drive (no GitHub):** set `REPO_URL = ""` and upload `DermAssist-AI-code.zip` to the top of your Google Drive ("My Drive"). Re-running this cell unpacks the zip again.

`sh()` and `py()` run commands and **stop the notebook if a command fails**, so an error is never silently skipped.

In [ ]:
import os, subprocess, sys
from pathlib import Path

def sh(*cmd):
    """Run a shell command, show its output, and raise if it fails."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {' '.join(cmd)}")

def py(*args):
    """Run Python (the notebook's own interpreter) with the given arguments."""
    sh(sys.executable, *args)

REPO_URL = "https://github.com/SatyaSreejay/DermAssistAI.git"  # set to "" to use the code zip instead
CODE_ZIP = Path("/content/drive/MyDrive/DermAssist-AI-code.zip")
REPO_DIR = Path("/content/DermAssist-AI")

if REPO_URL:
    if (REPO_DIR / ".git").exists():
        sh("git", "-C", REPO_DIR, "pull", "-q")
    else:
        sh("git", "clone", "-q", REPO_URL, REPO_DIR)
elif CODE_ZIP.exists():
    sh("unzip", "-q", "-o", CODE_ZIP, "-d", REPO_DIR.parent)  # the zip contains a DermAssist-AI/ folder
else:
    raise FileNotFoundError(f"REPO_URL is empty and no code zip was found at {CODE_ZIP}")
os.chdir(REPO_DIR)
py("-m", "pip", "install", "-q", "-r", "requirements-core.txt")
print("Working in", Path.cwd())

## 4. Run the tests (synthetic data only)

In [ ]:
py("-m", "pytest", "-q")

## 5. Point at the data

* `DRIVE_DATA`: the Drive folder with `HAM10000_metadata.csv` and the images (any sub-folder layout).
* `DRIVE_OUT`: Colab's disk is wiped when the session ends, so outputs that must survive (the split, reports, checkpoints) go to Drive.
* Reading ~10,000 small files straight from Drive is slow. If the zip is on Drive, set `ZIP_ON_DRIVE`; it is copied and unzipped once to the fast local disk.

In [ ]:
DRIVE_DATA = Path("/content/drive/MyDrive/HAM10000")          # TODO: adjust
DRIVE_OUT  = Path("/content/drive/MyDrive/DermAssist-AI")    # outputs that must survive the session
ZIP_ON_DRIVE = None  # e.g. DRIVE_DATA / "archive.zip"

IMAGE_DIR = DRIVE_DATA
if ZIP_ON_DRIVE:
    LOCAL = Path("/content/ham")
    if not (LOCAL / ".unzipped").exists():
        LOCAL.mkdir(parents=True, exist_ok=True)
        sh("cp", ZIP_ON_DRIVE, LOCAL / "data.zip")
        sh("unzip", "-q", "-n", LOCAL / "data.zip", "-d", LOCAL)
        (LOCAL / "data.zip").unlink()
        (LOCAL / ".unzipped").touch()
    IMAGE_DIR = LOCAL

# Use the metadata on Drive if present, otherwise look for it inside the image folder.
METADATA = DRIVE_DATA / "HAM10000_metadata.csv"
if not METADATA.exists():
    found = sorted(IMAGE_DIR.rglob("HAM10000_metadata.csv"))
    METADATA = found[0] if found else METADATA
PROCESSED, REPORTS = DRIVE_OUT / "processed", DRIVE_OUT / "reports"
if not METADATA.exists():
    raise FileNotFoundError(f"HAM10000_metadata.csv not found in {DRIVE_DATA} or {IMAGE_DIR}")
if not IMAGE_DIR.exists():
    raise FileNotFoundError(f"Image folder not found: {IMAGE_DIR}")
print("metadata:", METADATA)
print("images:  ", IMAGE_DIR)

## 6. Training paths
`SPLITS` is the split made by the quickstart notebook. Checkpoints go to Drive so they survive the session.

In [ ]:
SPLITS = PROCESSED / "splits.csv"
ARTIFACTS = DRIVE_OUT / "artifacts" / "baseline"
if not SPLITS.exists():
    raise FileNotFoundError(f"{SPLITS} not found. Run colab_quickstart.ipynb first.")
COMMON = ["--splits", SPLITS, "--image-dir", IMAGE_DIR, "--output-dir", ARTIFACTS]
print("splits:", SPLITS, "\nartifacts:", ARTIFACTS)

## 7. Preflight
Checks the config, the split and that every training and validation image can be found. Trains nothing.

In [ ]:
py("-m", "scripts.train_baseline", "--check-only", *COMMON)

## 8. Smoke run

Two batches per epoch and one epoch per stage, to prove the whole loop (GPU, data loading, checkpoint saving) works. Downloads the ImageNet ResNet18 weights (about 45 MB) the first time. **Its numbers are not results.**

In [ ]:
py("-m", "scripts.train_baseline", "--smoke", *COMMON)

## 9. Full training

Runs the stages in `configs/baseline.yaml` (head only, then fine-tuning `layer4`) with early stopping on validation macro-F1. Leave `RUN_FULL_TRAINING = False` until the preflight and smoke run look right.

To compare imbalance strategies, run it again with `"--class-weighting", "none"` added; each setting gets its own run folder.

In [ ]:
RUN_FULL_TRAINING = False  # set to True to start training

if RUN_FULL_TRAINING:
    py("-m", "scripts.train_baseline", *COMMON)
else:
    print("Full training skipped (RUN_FULL_TRAINING is False).")

## 10. Evaluate the trained model on the validation split
The run folder name is printed at the end of training.

In [ ]:
RUN_NAME = "resnet18_train_100pct_balanced_seed42"  # adjust if you changed subset, weighting or seed
CHECKPOINT = ARTIFACTS / RUN_NAME / "best.pt"
if CHECKPOINT.exists():
    py("-m", "scripts.evaluate", "--checkpoint", CHECKPOINT, "--split", "val",
       "--splits", SPLITS, "--image-dir", IMAGE_DIR)
else:
    print(f"No checkpoint at {CHECKPOINT} yet.")

## 11. Look at the training curve and validation results

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

run = ARTIFACTS / RUN_NAME
if (run / "history.csv").exists():
    display(pd.read_csv(run / "history.csv"))
    display(pd.read_csv(run / "eval_val" / "per_class.csv", index_col=0).round(3))
    display(Image(str(run / "eval_val" / "confusion_matrix.png"), width=550))